# Imports

In [26]:
from pathlib import Path
import json
from collections import Counter, defaultdict

import pandas as pd
from IPython.display import display
from datasets import get_dataset_config_names, load_dataset
from huggingface_hub import hf_hub_download
import zipfile
import io
import csv

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 200)
pd.set_option("display.width", 200)

In [27]:
DATASET_NAME = "haonan-li/cmmlu"

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = Path("/scratch/workspace/emma/standard-project-m2-mystery-machine")

RAW_DIR = REPO_ROOT / "multilingual" / "data" / "raw" / "cmmlu"
PROCESSED_DIR = REPO_ROOT / "multilingual" / "data" / "processed" / "cmmlu"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

ALL_PATH = PROCESSED_DIR / "cmmlu_chinese_all.jsonl"
EVAL_PATH = PROCESSED_DIR / "cmmlu_chinese_eval.jsonl"
TRAIN_PATH = PROCESSED_DIR / "cmmlu_chinese_train.jsonl"

print("RAW_DIR:      ", RAW_DIR)
print("PROCESSED_DIR:", PROCESSED_DIR)
print("ALL_PATH:     ", ALL_PATH)
print("EVAL_PATH:    ", EVAL_PATH)
print("TRAIN_PATH:   ", TRAIN_PATH)

OUTPUT_PATH = RAW_DIR / "cmmlu.jsonl"

VALID_ANSWERS = {"A", "B", "C", "D"}

print("NOTEBOOK_DIR:", NOTEBOOK_DIR)
print("REPO_ROOT:   ", REPO_ROOT)
print("RAW_DIR:     ", RAW_DIR)
print("OUTPUT_PATH: ", OUTPUT_PATH)

RAW_DIR:       /scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/data/raw/cmmlu
PROCESSED_DIR: /scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/data/processed/cmmlu
ALL_PATH:      /scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/data/processed/cmmlu/cmmlu_chinese_all.jsonl
EVAL_PATH:     /scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/data/processed/cmmlu/cmmlu_chinese_eval.jsonl
TRAIN_PATH:    /scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/data/processed/cmmlu/cmmlu_chinese_train.jsonl
NOTEBOOK_DIR: /scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/notebooks
REPO_ROOT:    /scratch/workspace/emma/standard-project-m2-mystery-machine
RAW_DIR:      /scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/data/raw/cmmlu
OUTPUT_PATH:  /scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/data/raw/cmmlu/cmmlu.jsonl

In [28]:
ZIP_NAME = "cmmlu_v1_0_1.zip"

zip_path = hf_hub_download(
    repo_id=DATASET_NAME,
    filename=ZIP_NAME,
    repo_type="dataset",
)

print("Downloaded zip path:")
print(zip_path)

with zipfile.ZipFile(zip_path) as z:
    all_names = z.namelist()
    csv_names = sorted([n for n in all_names if n.lower().endswith(".csv")])

print(f"Total files in zip: {len(all_names)}")
print(f"CSV files in zip:   {len(csv_names)}")

for name in csv_names[:20]:
    print(name)

Downloaded zip path:
/scratch/hf_cache/hub/datasets--haonan-li--cmmlu/snapshots/efcc940752ea4a1ea94d2727f11f83858d64fc8e/cmmlu_v1_0_1.zip
Total files in zip: 137
CSV files in zip:   134
dev/agronomy.csv
dev/anatomy.csv
dev/ancient_chinese.csv
dev/arts.csv
dev/astronomy.csv
dev/business_ethics.csv
dev/chinese_civil_service_exam.csv
dev/chinese_driving_rule.csv
dev/chinese_food_culture.csv
dev/chinese_foreign_policy.csv
dev/chinese_history.csv
dev/chinese_literature.csv
dev/chinese_teacher_qualification.csv
dev/clinical_knowledge.csv
dev/college_actuarial_science.csv
dev/college_education.csv
dev/college_engineering_hydrology.csv
dev/college_law.csv
dev/college_mathematics.csv
dev/college_medical_statistics.csv


In [29]:
csv_structure_rows = []

for name in csv_names:
    path = Path(name)
    split = path.parts[0]
    subject = path.stem

    csv_structure_rows.append({
        "path": name,
        "split": split,
        "subject": subject,
    })

csv_structure_df = pd.DataFrame(csv_structure_rows)

print("Splits:")
display(csv_structure_df["split"].value_counts().sort_index().to_frame("n_csv_files"))

print("Number of subjects:")
display(csv_structure_df.groupby("split")["subject"].nunique().to_frame("n_subjects"))

print("Subjects missing in some split:")
subjects_by_split = {
    split: set(g["subject"])
    for split, g in csv_structure_df.groupby("split")
}

all_subjects = sorted(set().union(*subjects_by_split.values()))
for split in sorted(subjects_by_split):
    missing = sorted(set(all_subjects) - subjects_by_split[split])
    print(f"{split}: missing {len(missing)} subjects")
    if missing:
        print(missing)

print("\nFirst 30 subjects:")
for subject in all_subjects[:30]:
    print(subject)

Splits:


,n_csv_files
split,
dev,67
test,67


Number of subjects:


,n_subjects
split,
dev,67
test,67


Subjects missing in some split:
dev: missing 0 subjects
test: missing 0 subjects

First 30 subjects:
agronomy
anatomy
ancient_chinese
arts
astronomy
business_ethics
chinese_civil_service_exam
chinese_driving_rule
chinese_food_culture
chinese_foreign_policy
chinese_history
chinese_literature
chinese_teacher_qualification
clinical_knowledge
college_actuarial_science
college_education
college_engineering_hydrology
college_law
college_mathematics
college_medical_statistics
college_medicine
computer_science
computer_security
conceptual_physics
construction_project_management
economics
education
electrical_engineering
elementary_chinese
elementary_commonsense


In [30]:
file_stats = []

with zipfile.ZipFile(zip_path) as z:
    for name in csv_names:
        with z.open(name) as f:
            text_f = io.TextIOWrapper(f, encoding="utf-8-sig")
            reader = csv.DictReader(text_f)
            rows = list(reader)

        file_stats.append({
            "path": name,
            "split": Path(name).parts[0],
            "subject": Path(name).stem,
            "fieldnames": tuple(reader.fieldnames or []),
            "n_rows": len(rows),
        })

file_stats_df = pd.DataFrame(file_stats)

print("Unique CSV headers:")
display(
    file_stats_df["fieldnames"]
    .value_counts()
    .reset_index()
    .rename(columns={"index": "fieldnames", "fieldnames": "n_files"})
)

print("\nRows by split:")
display(
    file_stats_df.groupby("split")["n_rows"]
    .agg(["count", "sum", "min", "max", "mean"])
    .sort_index()
)

print("\nSmallest files:")
display(file_stats_df.sort_values("n_rows").head(20))

print("\nLargest files:")
display(file_stats_df.sort_values("n_rows", ascending=False).head(20))

Unique CSV headers:


,n_files,count
0,"(, Question, A, B, C, D, Answer)",134



Rows by split:


,count,sum,min,max,mean
split,,,,,
dev,67,335,5,5,5.000000
test,67,11582,105,411,172.865672



Smallest files:


,path,split,subject,fieldnames,n_rows
0,dev/agronomy.csv,dev,agronomy,"(, Question, A, B, C, D, Answer)",5
1,dev/anatomy.csv,dev,anatomy,"(, Question, A, B, C, D, Answer)",5
2,dev/ancient_chinese.csv,dev,ancient_chinese,"(, Question, A, B, C, D, Answer)",5
3,dev/arts.csv,dev,arts,"(, Question, A, B, C, D, Answer)",5
4,dev/astronomy.csv,dev,astronomy,"(, Question, A, B, C, D, Answer)",5
5,dev/business_ethics.csv,dev,business_ethics,"(, Question, A, B, C, D, Answer)",5
6,dev/chinese_civil_service_exam.csv,dev,chinese_civil_service_exam,"(, Question, A, B, C, D, Answer)",5
7,dev/chinese_driving_rule.csv,dev,chinese_driving_rule,"(, Question, A, B, C, D, Answer)",5
8,dev/chinese_food_culture.csv,dev,chinese_food_culture,"(, Question, A, B, C, D, Answer)",5
9,dev/chinese_foreign_policy.csv,dev,chinese_foreign_policy,"(, Question, A, B, C, D, Answer)",5



Largest files:


,path,split,subject,fieldnames,n_rows
112,test/jurisprudence.csv,test,jurisprudence,"(, Question, A, B, C, D, Answer)",411
124,test/professional_medicine.csv,test,professional_medicine,"(, Question, A, B, C, D, Answer)",376
77,test/chinese_history.csv,test,chinese_history,"(, Question, A, B, C, D, Answer)",323
87,test/college_medicine.csv,test,college_medicine,"(, Question, A, B, C, D, Answer)",273
95,test/elementary_chinese.csv,test,elementary_chinese,"(, Question, A, B, C, D, Answer)",252
97,test/elementary_information_and_technology.csv,test,elementary_information_and_technology,"(, Question, A, B, C, D, Answer)",238
80,test/clinical_knowledge.csv,test,clinical_knowledge,"(, Question, A, B, C, D, Answer)",237
125,test/professional_psychology.csv,test,professional_psychology,"(, Question, A, B, C, D, Answer)",232
98,test/elementary_mathematics.csv,test,elementary_mathematics,"(, Question, A, B, C, D, Answer)",230
128,test/sociology.csv,test,sociology,"(, Question, A, B, C, D, Answer)",226


In [31]:
file_stats = []

with zipfile.ZipFile(zip_path) as z:
    for name in csv_names:
        with z.open(name) as f:
            text_f = io.TextIOWrapper(f, encoding="utf-8-sig")
            reader = csv.DictReader(text_f)
            rows = list(reader)

        file_stats.append({
            "path": name,
            "split": Path(name).parts[0],
            "subject": Path(name).stem,
            "fieldnames": tuple(reader.fieldnames or []),
            "n_rows": len(rows),
        })

file_stats_df = pd.DataFrame(file_stats)

print("Unique CSV headers:")
display(
    file_stats_df["fieldnames"]
    .value_counts()
    .reset_index()
    .rename(columns={"index": "fieldnames", "fieldnames": "n_files"})
)

print("\nRows by split:")
display(
    file_stats_df.groupby("split")["n_rows"]
    .agg(["count", "sum", "min", "max", "mean"])
    .sort_index()
)

print("\nSmallest files:")
display(file_stats_df.sort_values("n_rows").head(20))

print("\nLargest files:")
display(file_stats_df.sort_values("n_rows", ascending=False).head(20))

Unique CSV headers:


,n_files,count
0,"(, Question, A, B, C, D, Answer)",134



Rows by split:


,count,sum,min,max,mean
split,,,,,
dev,67,335,5,5,5.000000
test,67,11582,105,411,172.865672



Smallest files:


,path,split,subject,fieldnames,n_rows
0,dev/agronomy.csv,dev,agronomy,"(, Question, A, B, C, D, Answer)",5
1,dev/anatomy.csv,dev,anatomy,"(, Question, A, B, C, D, Answer)",5
2,dev/ancient_chinese.csv,dev,ancient_chinese,"(, Question, A, B, C, D, Answer)",5
3,dev/arts.csv,dev,arts,"(, Question, A, B, C, D, Answer)",5
4,dev/astronomy.csv,dev,astronomy,"(, Question, A, B, C, D, Answer)",5
5,dev/business_ethics.csv,dev,business_ethics,"(, Question, A, B, C, D, Answer)",5
6,dev/chinese_civil_service_exam.csv,dev,chinese_civil_service_exam,"(, Question, A, B, C, D, Answer)",5
7,dev/chinese_driving_rule.csv,dev,chinese_driving_rule,"(, Question, A, B, C, D, Answer)",5
8,dev/chinese_food_culture.csv,dev,chinese_food_culture,"(, Question, A, B, C, D, Answer)",5
9,dev/chinese_foreign_policy.csv,dev,chinese_foreign_policy,"(, Question, A, B, C, D, Answer)",5



Largest files:


,path,split,subject,fieldnames,n_rows
112,test/jurisprudence.csv,test,jurisprudence,"(, Question, A, B, C, D, Answer)",411
124,test/professional_medicine.csv,test,professional_medicine,"(, Question, A, B, C, D, Answer)",376
77,test/chinese_history.csv,test,chinese_history,"(, Question, A, B, C, D, Answer)",323
87,test/college_medicine.csv,test,college_medicine,"(, Question, A, B, C, D, Answer)",273
95,test/elementary_chinese.csv,test,elementary_chinese,"(, Question, A, B, C, D, Answer)",252
97,test/elementary_information_and_technology.csv,test,elementary_information_and_technology,"(, Question, A, B, C, D, Answer)",238
80,test/clinical_knowledge.csv,test,clinical_knowledge,"(, Question, A, B, C, D, Answer)",237
125,test/professional_psychology.csv,test,professional_psychology,"(, Question, A, B, C, D, Answer)",232
98,test/elementary_mathematics.csv,test,elementary_mathematics,"(, Question, A, B, C, D, Answer)",230
128,test/sociology.csv,test,sociology,"(, Question, A, B, C, D, Answer)",226


In [32]:
all_rows = []
failed_rows = []

REQUIRED_COLUMNS = {"Question", "A", "B", "C", "D", "Answer"}

with zipfile.ZipFile(zip_path) as z:
    for name in csv_names:
        split = Path(name).parts[0]
        subject = Path(name).stem

        with z.open(name) as f:
            text_f = io.TextIOWrapper(f, encoding="utf-8-sig")
            reader = csv.DictReader(text_f)
            fieldnames = reader.fieldnames or []

            missing_cols = sorted(REQUIRED_COLUMNS - set(fieldnames))
            if missing_cols:
                failed_rows.append({
                    "path": name,
                    "reason": f"missing_columns: {missing_cols}",
                    "fieldnames": fieldnames,
                })
                continue

            rows_in_file = list(reader)
            n_examples_in_subject_split = len(rows_in_file)

            for idx, row in enumerate(rows_in_file):
                all_rows.append({
                    "question": row.get("Question", ""),
                    "A": row.get("A", ""),
                    "B": row.get("B", ""),
                    "C": row.get("C", ""),
                    "D": row.get("D", ""),
                    "answer": row.get("Answer", ""),
                    "subject": subject,
                    "source_split": split,
                    "source_idx": idx,
                    "source_path": name,
                    "n_examples_in_subject_split": n_examples_in_subject_split,
                })

df_raw = pd.DataFrame(all_rows)

print(f"Loaded raw rows: {len(df_raw)}")
print(f"Failed files:    {len(failed_rows)}")

if failed_rows:
    display(pd.DataFrame(failed_rows))

display(df_raw.head())

Loaded raw rows: 11917
Failed files:    0


,question,A,B,C,D,answer,subject,source_split,source_idx,source_path,n_examples_in_subject_split
0,肉牛屠宰后，胴体的哪个部位肉质较好,胸,腹,大腿,小腿,C,agronomy,dev,0,dev/agronomy.csv,5
1,下列鸭品种中，产蛋量最高的品种是,高邮鸭,北京鸭,樱桃谷鸭,绍鸭,D,agronomy,dev,1,dev/agronomy.csv,5
2,常见家畜中输精管不形成输精管壶腹的是,马,猪,牛,羊,B,agronomy,dev,2,dev/agronomy.csv,5
3,牛的发情周期平均为,16～17天,21天,28天,19～20天,B,agronomy,dev,3,dev/agronomy.csv,5
4,羊胴体中，肉质较好的部位是,胸下肉,肩胛肉,后腿肉,小腿肉,C,agronomy,dev,4,dev/agronomy.csv,5


# Inspect

In [33]:
print("Raw shape:")
print(df_raw.shape)

print("\nRows by source split:")
display(df_raw["source_split"].value_counts().sort_index().to_frame("n_rows"))

print("\nRows by subject:")
display(df_raw["subject"].value_counts().sort_index().to_frame("n_rows"))

print("\nAnswer distribution before cleaning:")
display(df_raw["answer"].value_counts(dropna=False).sort_index().to_frame("n_rows"))

print("\nRandom raw examples:")
display(
    df_raw.sample(min(10, len(df_raw)), random_state=42)
    [["subject", "source_split", "question", "A", "B", "C", "D", "answer"]]
)

Raw shape:
(11917, 11)

Rows by source split:


,n_rows
source_split,
dev,335
test,11582



Rows by subject:


,n_rows
subject,
agronomy,174
anatomy,153
ancient_chinese,169
arts,165
astronomy,170
business_ethics,214
chinese_civil_service_exam,165
chinese_driving_rule,136
chinese_food_culture,141



Answer distribution before cleaning:


,n_rows
answer,
A,3019
B,2960
C,2949
D,2989



Random raw examples:


,subject,source_split,question,A,B,C,D,answer
3995,computer_security,test,信息隐藏主要研究如何将机密信息秘密隐藏于另一公开的信息中。以下关于利用多媒体数据来隐藏机密信息的叙述中，错误的是,人眼或人耳对某些信息有一定的掩蔽效应,多媒体信息本身有很大的冗余性,多媒体信息本身编码效率很高,信息嵌入到多媒体信息中不影响多媒体本身的传送和使用,C
7476,international_law,test,在合同的法律适用上，持“单一论”或“整体论”的学者们是主张,合同不分国内合同与国际合同，统一只适用同一法律,国际合同应适用的法律只能由国内法统一加以规定,把所有的合同的应适用法律由一条冲突规范规定,把合同的所有方面的问题交由一个法律解决,D
7302,human_sexuality,test,按照联合国1997年的定义，( )是一种战略，将妇女和男子的关注事项和经验作为一个整体，纳入政治、经济和社会等所有领域的政策和方针的设计、落实、监测和评估，使男女都能平等受益，中止不平等现象，最终实现男女两性平等的目标。,女性主流化,社会性别意识纳入决策主流,平权行动,女权主义,B
3148,college_law,test,甲到某银行办理业务发现李某使用 ATM 机后忘记拔卡，遂用李某的银行卡连续取款共 12000 元。李某收到取款提示短信后返回银行，要求正在取款的甲交还钱款，甲转身逃跑并将尾随追赶的李某打成轻微伤，甲的行为应认定为,侵占罪,信用卡诈骗罪,盗窃罪,抢劫罪,D
747,ancient_chinese,test,下列句子，属于被动句的一句是,故不能推車而及。,父母唯其疾之憂。,夫戰，勇氣也。,豕人立而啼。,A
4048,computer_security,test,攻击者通过对目标主机进行端口扫描，可以直接获得,目标主机的操作系统信息,目标主机的登录口令,目标主机的硬件设备信息,目标主机开放端口服务信息,D
4759,electrical_engineering,test,物体由于受外界影响（如升商、变形等）而其有的能f称为,机械能,惯能,势能,动能,C
6425,high_school_biology,test,下表中根据实验目的，所选用的试剂与预期的实验结果正确的是,实验目的:检测植物组织中的脂肪。试剂：双缩脲试剂。预期的实验结果：脂肪颗粒被染成红色。,实验目的:观察DNA和RNA在细胞中的分布。试剂：斐林试剂吡罗红。预期的实验结果：斐林试剂将DNA染成绿色，吡罗红将RNA染成红色。,实验目的:观察根尖分生组织细胞的有丝分裂。试剂：醋酸洋红。预期的实验结果：染色体被染成紫红色。,实验目的:检测植物组织中的葡萄糖。试剂：甲基绿。预期的实验结果：葡萄糖与甲基绿作用，生成绿色沉淀。,C
3819,computer_science,test,以下软件中，___不是操作系统软件,Windows xp,unix,linux,microsoft office,D
379,agronomy,test,下列生物中，能进行次级生产的是,小麦,兔,水稻,蔬菜,B


In [34]:
df = df_raw.copy()

TEXT_COLUMNS = [
    "question",
    "A",
    "B",
    "C",
    "D",
    "answer",
    "subject",
    "source_split",
    "source_path",
]

for col in TEXT_COLUMNS:
    df[col] = df[col].fillna("").astype(str).str.strip()

df["answer"] = df["answer"].str.upper()

print("Answer distribution before filtering:")
display(df["answer"].value_counts(dropna=False).sort_index().to_frame("n_rows"))

n_before_cleaning = len(df)

df = df[
    (df["question"] != "")
    & (df["A"] != "")
    & (df["B"] != "")
    & (df["C"] != "")
    & (df["D"] != "")
    & (df["answer"].isin(VALID_ANSWERS))
].copy()

df = df.reset_index(drop=True)

n_after_cleaning = len(df)

print(f"Rows before basic MCQ cleaning: {n_before_cleaning}")
print(f"Rows after basic MCQ cleaning:  {n_after_cleaning}")
print(f"Removed invalid rows:           {n_before_cleaning - n_after_cleaning}")

print("\nAnswer distribution after filtering:")
display(df["answer"].value_counts().sort_index().to_frame("n_rows"))

Answer distribution before filtering:


,n_rows
answer,
A,3019
B,2960
C,2949
D,2989


Rows before basic MCQ cleaning: 11917
Rows after basic MCQ cleaning:  11917
Removed invalid rows:           0

Answer distribution after filtering:


,n_rows
answer,
A,3019
B,2960
C,2949
D,2989


In [35]:
def infer_category(subject):
    s = str(subject)

    if any(k in s for k in [
        "mathematics", "physics", "chemistry", "biology", "computer",
        "machine_learning", "engineering", "electrical", "agronomy",
        "anatomy", "astronomy", "genetics", "virology", "medicine",
        "medical", "clinical", "nutrition", "food_science",
    ]):
        return "STEM"

    if any(k in s for k in [
        "law", "jurisprudence", "accounting", "management", "marketing",
        "business", "economics", "actuarial", "project_management",
        "teacher_qualification", "education",
    ]):
        return "Law & Professional"

    if any(k in s for k in [
        "history", "literature", "chinese", "arts", "philosophy",
        "world_religions", "ancient_chinese", "modern_chinese",
    ]):
        return "Humanities"

    if any(k in s for k in [
        "sociology", "ethnology", "politics", "foreign_policy",
        "marxist", "security", "public_relations", "human_sexuality",
        "psychology", "civil_service", "driving_rule", "moral",
    ]):
        return "Social Science"

    return "Other"


def infer_regionality(subject):
    s = str(subject)

    if any(k in s for k in [
        "chinese",
        "china",
        "civil_service",
        "driving_rule",
        "foreign_policy",
        "chinese_history",
        "modern_chinese",
        "ancient_chinese",
        "marxist_theory",
        "ethnology",
        "traditional_chinese_medicine",
        "college_law",
        "professional_law",
        "jurisprudence",
        "legal_and_moral_basis",
    ]):
        return "regional"

    if any(k in s for k in [
        "law",
        "accounting",
        "teacher_qualification",
        "education",
        "economics",
        "management",
        "marketing",
        "business",
        "medicine",
        "clinical",
        "psychology",
        "public_relations",
        "security",
        "sociology",
        "food_culture",
    ]):
        return "regional_implicit"

    return "region_agnostic"


df["category"] = df["subject"].apply(infer_category)
df["regionality"] = df["subject"].apply(infer_regionality)

print("Rows by category:")
display(df["category"].value_counts().sort_index().to_frame("n_rows"))

print("\nRows by regionality:")
display(df["regionality"].value_counts().sort_index().to_frame("n_rows"))

print("\nSubjects by category and regionality:")
display(
    df.groupby(["category", "regionality"])["subject"]
    .nunique()
    .reset_index(name="n_subjects")
    .sort_values(["category", "regionality"])
)

Rows by category:


,n_rows
category,
Humanities,2244
Law & Professional,2612
Other,1198
STEM,4244
Social Science,1619



Rows by regionality:


,n_rows
regionality,
region_agnostic,5171
regional,3310
regional_implicit,3436



Subjects by category and regionality:


,category,regionality,n_subjects
0,Humanities,region_agnostic,4
1,Humanities,regional,9
2,Law & Professional,region_agnostic,1
3,Law & Professional,regional,4
4,Law & Professional,regional_implicit,9
5,Other,region_agnostic,7
6,STEM,region_agnostic,19
7,STEM,regional,1
8,STEM,regional_implicit,4
9,Social Science,region_agnostic,2


In [36]:
subject_summary_df = (
    df.groupby(["subject", "category", "regionality"])
    .size()
    .reset_index(name="n_examples")
    .sort_values(["regionality", "category", "subject"])
)

display(subject_summary_df)

print("Regional subjects:")
for subject in sorted(subject_summary_df.loc[subject_summary_df["regionality"] == "regional", "subject"]):
    print("  -", subject)

print("\nRegional implicit subjects:")
for subject in sorted(subject_summary_df.loc[subject_summary_df["regionality"] == "regional_implicit", "subject"]):
    print("  -", subject)

print("\nRegion agnostic subjects:")
for subject in sorted(subject_summary_df.loc[subject_summary_df["regionality"] == "region_agnostic", "subject"]):
    print("  -", subject)

,subject,category,regionality,n_examples
3,arts,Humanities,region_agnostic,165
54,philosophy,Humanities,region_agnostic,110
65,world_history,Humanities,region_agnostic,166
66,world_religions,Humanities,region_agnostic,165
14,college_actuarial_science,Law & Professional,region_agnostic,111
29,elementary_commonsense,Other,region_agnostic,203
30,elementary_information_and_technology,Other,region_agnostic,243
35,global_facts,Other,region_agnostic,154
38,high_school_geography,Other,region_agnostic,123
44,journalism,Other,region_agnostic,177


Regional subjects:
  - ancient_chinese
  - chinese_civil_service_exam
  - chinese_driving_rule
  - chinese_food_culture
  - chinese_foreign_policy
  - chinese_history
  - chinese_literature
  - chinese_teacher_qualification
  - college_law
  - elementary_chinese
  - ethnology
  - jurisprudence
  - legal_and_moral_basis
  - marxist_theory
  - modern_chinese
  - professional_law
  - traditional_chinese_medicine

Regional implicit subjects:
  - business_ethics
  - clinical_knowledge
  - college_education
  - college_medicine
  - computer_security
  - construction_project_management
  - economics
  - education
  - international_law
  - management
  - marketing
  - professional_accounting
  - professional_medicine
  - professional_psychology
  - public_relations
  - security_study
  - sociology

Region agnostic subjects:
  - agronomy
  - anatomy
  - arts
  - astronomy
  - college_actuarial_science
  - college_engineering_hydrology
  - college_mathematics
  - college_medical_statistics
  - c

In [37]:
subject_summary_df = (
    df.groupby(["subject", "category", "regionality"])
    .size()
    .reset_index(name="n_examples")
    .sort_values(["regionality", "category", "subject"])
)

display(subject_summary_df)

print("Regional subjects:")
for subject in sorted(subject_summary_df.loc[subject_summary_df["regionality"] == "regional", "subject"]):
    print("  -", subject)

print("\nRegional implicit subjects:")
for subject in sorted(subject_summary_df.loc[subject_summary_df["regionality"] == "regional_implicit", "subject"]):
    print("  -", subject)

print("\nRegion agnostic subjects:")
for subject in sorted(subject_summary_df.loc[subject_summary_df["regionality"] == "region_agnostic", "subject"]):
    print("  -", subject)

,subject,category,regionality,n_examples
3,arts,Humanities,region_agnostic,165
54,philosophy,Humanities,region_agnostic,110
65,world_history,Humanities,region_agnostic,166
66,world_religions,Humanities,region_agnostic,165
14,college_actuarial_science,Law & Professional,region_agnostic,111
29,elementary_commonsense,Other,region_agnostic,203
30,elementary_information_and_technology,Other,region_agnostic,243
35,global_facts,Other,region_agnostic,154
38,high_school_geography,Other,region_agnostic,123
44,journalism,Other,region_agnostic,177


Regional subjects:
  - ancient_chinese
  - chinese_civil_service_exam
  - chinese_driving_rule
  - chinese_food_culture
  - chinese_foreign_policy
  - chinese_history
  - chinese_literature
  - chinese_teacher_qualification
  - college_law
  - elementary_chinese
  - ethnology
  - jurisprudence
  - legal_and_moral_basis
  - marxist_theory
  - modern_chinese
  - professional_law
  - traditional_chinese_medicine

Regional implicit subjects:
  - business_ethics
  - clinical_knowledge
  - college_education
  - college_medicine
  - computer_security
  - construction_project_management
  - economics
  - education
  - international_law
  - management
  - marketing
  - professional_accounting
  - professional_medicine
  - professional_psychology
  - public_relations
  - security_study
  - sociology

Region agnostic subjects:
  - agronomy
  - anatomy
  - arts
  - astronomy
  - college_actuarial_science
  - college_engineering_hydrology
  - college_mathematics
  - college_medical_statistics
  - c

In [38]:
# manual corrections of regionality based on subject name inspection
CATEGORY_OVERRIDES = {
    "college_actuarial_science": "STEM",
    "high_school_geography": "Social Science",
    "high_school_politics": "Social Science",
    "journalism": "Humanities",
    "sports_science": "Other",
}

REGIONALITY_OVERRIDES = {
    "high_school_politics": "regional",
    "high_school_geography": "regional",
    "journalism": "regional_implicit",
    "sports_science": "regional_implicit",
}

df["category"] = df["subject"].map(CATEGORY_OVERRIDES).fillna(df["category"])
df["regionality"] = df["subject"].map(REGIONALITY_OVERRIDES).fillna(df["regionality"])

print("Rows by category after overrides:")
display(df["category"].value_counts().sort_index().to_frame("n_rows"))

print("\nRows by regionality after overrides:")
display(df["regionality"].value_counts().sort_index().to_frame("n_rows"))

print("\nSubject summary after overrides:")
subject_summary_df = (
    df.groupby(["subject", "category", "regionality"])
    .size()
    .reset_index(name="n_examples")
    .sort_values(["regionality", "category", "subject"])
)
display(subject_summary_df)

Rows by category after overrides:


,n_rows
category,
Humanities,2421
Law & Professional,2501
Other,898
STEM,4355
Social Science,1742



Rows by regionality after overrides:


,n_rows
regionality,
region_agnostic,4553
regional,3581
regional_implicit,3783



Subject summary after overrides:


,subject,category,regionality,n_examples
3,arts,Humanities,region_agnostic,165
54,philosophy,Humanities,region_agnostic,110
65,world_history,Humanities,region_agnostic,166
66,world_religions,Humanities,region_agnostic,165
29,elementary_commonsense,Other,region_agnostic,203
30,elementary_information_and_technology,Other,region_agnostic,243
35,global_facts,Other,region_agnostic,154
47,logical,Other,region_agnostic,128
0,agronomy,STEM,region_agnostic,174
1,anatomy,STEM,region_agnostic,153


# Dediplication set

In [39]:
DEDUP_COLUMNS = ["subject", "question", "A", "B", "C", "D"]

conflict_counts = (
    df.groupby(DEDUP_COLUMNS)["answer"]
    .nunique()
    .reset_index(name="n_unique_answers")
)

conflicts = conflict_counts[conflict_counts["n_unique_answers"] > 1]

print(f"Conflicting duplicate groups: {len(conflicts)}")
display(conflicts.head(20))

if len(conflicts) > 0:
    conflict_keys = conflicts[DEDUP_COLUMNS]
    df_conflicts = df.merge(conflict_keys, on=DEDUP_COLUMNS, how="inner")
    df_conflicts = df_conflicts.sort_values(
        DEDUP_COLUMNS + ["source_split", "source_idx"]
    )
    display(df_conflicts.head(50))

Conflicting duplicate groups: 0


,subject,question,A,B,C,D,n_unique_answers


In [40]:
n_before_dedup = len(df)

df = df.drop_duplicates(subset=DEDUP_COLUMNS, keep="first").copy()
df = df.reset_index(drop=True)

n_after_dedup = len(df)

print(f"Rows before deduplication: {n_before_dedup}")
print(f"Rows after deduplication:  {n_after_dedup}")
print(f"Removed duplicates:        {n_before_dedup - n_after_dedup}")

duplicate_check = df.duplicated(subset=DEDUP_COLUMNS).sum()
print(f"Remaining duplicates by subject/question/options: {duplicate_check}")
assert duplicate_check == 0

Rows before deduplication: 11917
Rows after deduplication:  11915
Removed duplicates:        2
Remaining duplicates by subject/question/options: 0


In [41]:
SOURCE_TO_PROJECT_SPLIT = {
    "dev": "eval",
    "test": "train",
}

df["split"] = df["source_split"].map(SOURCE_TO_PROJECT_SPLIT)

missing_split = sorted(df.loc[df["split"].isna(), "source_split"].unique())
print("Missing project split mappings:", missing_split)
assert not missing_split

print("Rows by original source split:")
display(df["source_split"].value_counts().sort_index().to_frame("n_examples"))

print("\nRows by project split:")
display(df["split"].value_counts().sort_index().to_frame("n_examples"))

print("\nRows by project split and regionality:")
display(
    df.groupby(["split", "regionality"])
    .size()
    .reset_index(name="n_examples")
    .pivot(index="split", columns="regionality", values="n_examples")
    .fillna(0)
    .astype(int)
)

Missing project split mappings: []
Rows by original source split:


,n_examples
source_split,
dev,335
test,11580



Rows by project split:


,n_examples
split,
eval,335
train,11580



Rows by project split and regionality:


regionality,region_agnostic,regional,regional_implicit
split,,,
eval,145,95,95
train,4406,3486,3688


In [42]:
def make_prompt(row):
    return (
        f"{row['question']}\n\n"
        f"A) {row['A']}\n"
        f"B) {row['B']}\n"
        f"C) {row['C']}\n"
        f"D) {row['D']}"
    )


def safe_id_part(text):
    return (
        str(text)
        .strip()
        .replace("/", "_")
        .replace("\\", "_")
        .replace(" ", "_")
        .replace("-", "_")
        .replace(":", "_")
    )


def make_record(row, idx):
    subject = safe_id_part(row["subject"])
    project_split = safe_id_part(row["split"])
    source_split = safe_id_part(row["source_split"])
    source_idx = int(row["source_idx"])

    return {
        "id": f"cmmlu_chinese_{project_split}_{subject}_{source_split}_{source_idx}",
        "dataset": "cmmlu",
        "language": "chinese",
        "split": row["split"],
        "task_type": "mcq",
        "prompt": make_prompt(row),
        "answer": row["answer"],
        "metadata": {
            "subject": row["subject"],
            "category": row["category"],
            "regionality": row["regionality"],
            "source_split": row["source_split"],
            "source_idx": source_idx,
            "source_path": row["source_path"],
            "n_examples_in_subject_split": int(row["n_examples_in_subject_split"]),
        },
    }


records = []
for idx, row in df.reset_index(drop=True).iterrows():
    records.append(make_record(row, idx))

print(f"Created records: {len(records)}")
print(json.dumps(records[0], ensure_ascii=False, indent=2))

Created records: 11915
{
  "id": "cmmlu_chinese_eval_agronomy_dev_0",
  "dataset": "cmmlu",
  "language": "chinese",
  "split": "eval",
  "task_type": "mcq",
  "prompt": "肉牛屠宰后，胴体的哪个部位肉质较好\n\nA) 胸\nB) 腹\nC) 大腿\nD) 小腿",
  "answer": "C",
  "metadata": {
    "subject": "agronomy",
    "category": "STEM",
    "regionality": "region_agnostic",
    "source_split": "dev",
    "source_idx": 0,
    "source_path": "dev/agronomy.csv",
    "n_examples_in_subject_split": 5
  }
}


In [43]:
VALID_SPLITS = {"train", "eval"}
VALID_CATEGORIES = {
    "STEM",
    "Humanities",
    "Social Science",
    "Law & Professional",
    "Other",
}
VALID_REGIONALITIES = {
    "regional",
    "regional_implicit",
    "region_agnostic",
}


def validate_record(record):
    expected_keys = {
        "id",
        "dataset",
        "language",
        "split",
        "task_type",
        "prompt",
        "answer",
        "metadata",
    }

    assert set(record.keys()) == expected_keys

    assert isinstance(record["id"], str)
    assert record["id"].startswith("cmmlu_chinese_")
    assert record["dataset"] == "cmmlu"
    assert record["language"] == "chinese"
    assert record["split"] in VALID_SPLITS
    assert record["task_type"] == "mcq"
    assert record["answer"] in VALID_ANSWERS

    prompt = record["prompt"]
    assert isinstance(prompt, str)
    assert prompt.strip()
    assert "\n\nA) " in prompt
    assert "\nB) " in prompt
    assert "\nC) " in prompt
    assert "\nD) " in prompt
    assert "\\boxed" not in prompt

    metadata = record["metadata"]
    assert set(metadata.keys()) == {
        "subject",
        "category",
        "regionality",
        "source_split",
        "source_idx",
        "source_path",
        "n_examples_in_subject_split",
    }

    assert metadata["category"] in VALID_CATEGORIES
    assert metadata["regionality"] in VALID_REGIONALITIES
    assert metadata["source_split"] in {"dev", "test"}
    assert isinstance(metadata["source_idx"], int)
    assert isinstance(metadata["n_examples_in_subject_split"], int)


ids = [record["id"] for record in records]
assert len(ids) == len(set(ids)), "Duplicate record ids found."

for record in records:
    validate_record(record)

print("All CMMLU records validated successfully.")

All CMMLU records validated successfully.


# Save

In [47]:
def write_jsonl(records, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")


# Full all-in-one processed CMMLU file
write_jsonl(records, ALL_PATH)
print(f"Saved {len(records)} CMMLU records to:")
print(ALL_PATH)


# Train/eval processed files only
eval_records = [record for record in records if record["split"] == "eval"]
train_records = [record for record in records if record["split"] == "train"]

write_jsonl(eval_records, EVAL_PATH)
write_jsonl(train_records, TRAIN_PATH)

print(f"Saved {len(eval_records)} eval records to:")
print(EVAL_PATH)

print(f"Saved {len(train_records)} train records to:")
print(TRAIN_PATH)

Saved 11915 CMMLU records to:
/scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/data/processed/cmmlu/cmmlu_chinese_all.jsonl
Saved 335 eval records to:
/scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/data/processed/cmmlu/cmmlu_chinese_eval.jsonl
Saved 11580 train records to:
/scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/data/processed/cmmlu/cmmlu_chinese_train.jsonl


In [45]:
def read_jsonl(path):
    loaded = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            loaded.append(json.loads(line))
    return loaded


loaded_records = read_jsonl(OUTPUT_PATH)
print(f"Reloaded records from all file: {len(loaded_records)}")
assert len(loaded_records) == len(records)

for path in [OUTPUT_PATH] + list(SPLIT_TO_PATH.values()):
    loaded = read_jsonl(path)
    print(f"\n{path.name}: {len(loaded)}")

    ids = [record["id"] for record in loaded]
    assert len(ids) == len(set(ids)), f"Duplicate ids in {path}"

    for record in loaded:
        validate_record(record)

print("\nAll saved CMMLU files are valid.")

Reloaded records from all file: 11915

cmmlu.jsonl: 11915

cmmlu_chinese_eval.jsonl: 335

cmmlu_chinese_train.jsonl: 11580

All saved CMMLU files are valid.


In [46]:
summary_df = pd.DataFrame(
    {
        "id": record["id"],
        "language": record["language"],
        "split": record["split"],
        "answer": record["answer"],
        "subject": record["metadata"]["subject"],
        "category": record["metadata"]["category"],
        "regionality": record["metadata"]["regionality"],
        "source_split": record["metadata"]["source_split"],
        "source_path": record["metadata"]["source_path"],
    }
    for record in loaded_records
)

print("Final CMMLU summary")
print("-------------------")
print(f"Saved all records: {len(loaded_records)}")
print(f"Output path:       {OUTPUT_PATH}")

print("\nBy project split:")
display(summary_df["split"].value_counts().sort_index().to_frame("n_examples"))

print("\nBy original source split:")
display(summary_df["source_split"].value_counts().sort_index().to_frame("n_examples"))

print("\nBy answer:")
display(summary_df["answer"].value_counts().sort_index().to_frame("n_examples"))

print("\nBy category:")
display(summary_df["category"].value_counts().sort_index().to_frame("n_examples"))

print("\nBy regionality:")
display(summary_df["regionality"].value_counts().sort_index().to_frame("n_examples"))

print("\nBy split and regionality:")
display(
    summary_df.groupby(["split", "regionality"])
    .size()
    .reset_index(name="n_examples")
    .pivot(index="split", columns="regionality", values="n_examples")
    .fillna(0)
    .astype(int)
)

print("\nBy subject:")
display(summary_df["subject"].value_counts().sort_index().to_frame("n_examples"))

Final CMMLU summary
-------------------
Saved all records: 11915
Output path:       /scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/data/raw/cmmlu/cmmlu.jsonl

By project split:


,n_examples
split,
eval,335
train,11580



By original source split:


,n_examples
source_split,
dev,335
test,11580



By answer:


...

,n_examples
regionality,
region_agnostic,4551
regional,3581
regional_implicit,3783



By split and regionality:


regionality,region_agnostic,regional,regional_implicit
split,,,
eval,145,95,95
train,4406,3486,3688



By subject:


,n_examples
subject,
agronomy,174
anatomy,152
ancient_chinese,169
arts,165
astronomy,170
business_ethics,214
chinese_civil_service_exam,165
chinese_driving_rule,136
chinese_food_culture,141
